# AHR_RL: train a pre-race greening agent on the full Betfair recordings

This notebook runs the `ahr_rl` package end-to-end on **all** recordings in
`My Drive/betfair stream data/recordings`:

1. compile each `*.ndjson.gz` into a compact tape (cached back to Drive)
2. sanity baselines (do-nothing / random / rule-based scalper)
3. optional: planted-edge sanity check on synthetic races
4. PPO training (GPU if available)
5. held-out TEST evaluation vs baselines

Runtime: GPU (T4 is fine). The env simulation is CPU-bound, so more CPU cores help more than a bigger GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/betfair stream data'

In [ ]:
!git clone -b claude/wizardly-clarke-rb37hl https://github.com/tristanhowells/AHR_RL.git /content/AHR_RL || (cd /content/AHR_RL && git pull)
%cd /content/AHR_RL
!pip -q install orjson gymnasium>=1.0
!python -m pytest -q tests/test_exchange.py

## 1. Compile recordings -> tapes (incremental; cached on Drive)

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
TAPES = f'{DRIVE}/rl_tapes_v1'
os.makedirs(TAPES, exist_ok=True)
!python -m ahr_rl.tape "{DRIVE}/recordings" "{TAPES}" --workers {NCPU}
# local copy is much faster to read than the Drive mount
!mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
!ls /content/tapes | wc -l

## 2. Baselines on the most recent 150 races

In [ ]:
import glob
from ahr_rl.evaluate import evaluate, summarise, noop_policy, make_random_policy, make_scalper_policy
tapes = sorted(glob.glob('/content/tapes/*.npz'))
recent = tapes[-150:]
for name, pol in [('do_nothing', noop_policy), ('random_1pct', make_random_policy(0.01)), ('scalper_rule', make_scalper_policy())]:
    print(name, {k: round(v, 2) for k, v in summarise(evaluate(pol, recent)).items()})

## 2b. Edge test: is there anything to trade? (run this before long RL runs)
Supervised check on the same features and date split as the agent: do they predict 10/30/60s price moves
well enough to beat the round-trip cost? One rule per family is picked on VALIDATION days and scored once
on TEST days. ~10-20 min on the full dataset; results land in `{DRIVE}/rl_runs_ahr_rl/edge`.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!pip -q install scikit-learn
!python -m ahr_rl.edge --tapes "/content/tapes/*.npz" --out "{EDGE}" --workers {NCPU}

## 2c. Feature study: do WoM / matched-price range / WAP / trade flow add signal?
Univariate signal per feature, ablation (agent features vs + engineered vs engineered only) and permutation
importance. ~20-30 min on the full dataset; results in `{DRIVE}/rl_runs_ahr_rl/features`.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.feature_study --tapes "/content/tapes/*.npz" --out "{FEAT}" --workers {NCPU}

## 2d. Signal bot: does the passive-entry signal survive realistic execution?
Trains the edge-test models on train days, then trades them inside the full simulator (latency, queue position,
funds, auto-green, commission). Horizon and threshold are chosen on validation races; the chosen config is scored
once on test races. Reuses the samples from cell 2b. Results in `{DRIVE}/rl_runs_ahr_rl/signal_bot`.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.signal_bot --tapes "/content/tapes/*.npz" --out "{BOT}" --samples "{EDGE}/samples_base.parquet" --workers {NCPU}

## 2e. Fill-mode diagnosis: is the edge-test profit an execution artefact?
Reuses the models trained in 2d and replays the same signal on the TEST races under three fill models:
`realistic` (queue behind shown size), `no_queue` (always first in queue), `touch` (any trade at our price fills
the whole order = the edge test's assumption). ~10-20 min.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.signal_bot --tapes "/content/tapes/*.npz" --out "{BOT}" --diagnose --models "{BOT}/models_base.pkl" --horizons 10,30,60 --top-pcts 0.5,1 --workers {NCPU}

## 2f. Event study: jumps and WAP displacement. Does the price keep going or come back, and can you trade it?
Two event types, one method, on every race:
* **Jumps**: the price moves >= 3 or >= 5 ticks within 5s (the race deep dives showed big moves are jumps, not trends).
* **WAP displacement**: the price sits >= 3 or >= 5 ticks away from its session volume-weighted average matched price.
  A small sample suggested drifters above their WAP come partly back ("WAP drift, fade" = back it, green later).

Only tradeable runners (spread <= 3 ticks, price <= 30). For each event it measures the move over the next 5s-5min and to
the scheduled start, and what following or fading it would have paid with latency, real book depth and commission,
against a random-time control. Rules are picked on TRAIN days and scored once on unseen days, separately per event type.
Needs only the tapes from cell 1. ~5-15 min.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
JUMPS = f'{DRIVE}/rl_runs_ahr_rl/jumps'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
!python -m ahr_rl.jump_study --tapes "/content/tapes/*.npz" --out "{JUMPS}" --workers {NCPU}
from IPython.display import Image, display
display(Image(f'{JUMPS}/jump_curves.png'))

## 2g. Market study: tradeability, signals, and how they cross
Every 10s before the off, every runner is sampled with its market share (implied probability), favouritism rank,
time to the scheduled start (and to the actual off, hindsight only), WoM, WAP gaps, price rate of change (30s, 120s)
and volume rate of change (rate, acceleration, share of money vs share of market). Sections:
* **A** tradeability by market share, rank and time (cost, depth, typical move vs cost, hit rate needed to break even)
* **B** how well each signal predicts the next 30s / 120s / to-start / to-off move
* **C** crossed features: each signal's strength inside market-share, time and busy/quiet segments, and share x time
* **D** how the signals relate to each other
* **E** can you trade it: every signal x horizon x segment cell, picked on TRAIN days, scored on unseen days
* **F** all signals and segments crossed at once in boosted-tree models of real round-trip P&L, traded on unseen days

Needs only the tapes from cell 1. Samples are cached, so a re-run only redoes the analysis. ~15-40 min.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
MKT = f'{DRIVE}/rl_runs_ahr_rl/market'
NCPU = os.cpu_count()
if not os.path.isdir('/content/tapes') or not os.listdir('/content/tapes'):
    !mkdir -p /content/tapes && rsync -a "{TAPES}/" /content/tapes/
!python -m ahr_rl.market_study --tapes "/content/tapes/*.npz" --out "{MKT}" --workers {NCPU}
from IPython.display import Image, display
for f in ('tradeability.png', 'signal_by_time.png', 'crossed.png'):
    display(Image(f'{MKT}/{f}'))

## 3. (optional) Planted-edge sanity check
The agent should learn to profit on synthetic races where one runner steadily shortens. If it can't, fix the learner before blaming the market.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
RUN_SYNTHETIC = False
if RUN_SYNTHETIC:
    !python -m ahr_rl.synthetic /content/synthetic 200
    !python -m ahr_rl.train --tapes "/content/synthetic/*.npz" --out /content/runs/synthetic --total-steps 500000 --n-envs {NCPU} --eval-every 25

## 4. Train on the real recordings
Chronological split: oldest ~70% of race days train, next ~15% validation (checkpoint selection), newest ~15% test (untouched until step 5).

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.train --tapes "/content/tapes/*.npz" --out "{RUN}" --total-steps 5000000 --n-envs {max(4, NCPU)} --n-steps 256 --eval-every 25 --eval-races 200

In [ ]:
import pandas as pd
log = pd.read_csv(f'{RUN}/train_log.csv')
ax = log.plot(x='steps', y=['ep_green_mean'], figsize=(10, 4), title='train: mean worst-case $ per race (last 50)')
log.dropna(subset=['val_green']).plot(x='steps', y=['val_green'], ax=ax, marker='o')

## 5. Held-out test evaluation
`mean_green_$` = average guaranteed (worst-case, after commission) profit per race. `do_nothing` scores exactly 0, so the agent has to beat 0 on unseen races to be worth running.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
!python -m ahr_rl.report --run "{RUN}" --split test

## 5b. (optional) Challenger: continuous allocation spec, SAC vs PPO
See `docs/COMPARISON.md`. SAC benefits a lot from the GPU (many gradient steps).

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
BOT = f'{DRIVE}/rl_runs_ahr_rl/signal_bot'
FEAT = f'{DRIVE}/rl_runs_ahr_rl/features'
RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_v1'
NCPU = os.cpu_count()
RUN_CHALLENGERS = False
if RUN_CHALLENGERS:
    SAC_RUN = f'{DRIVE}/rl_runs_ahr_rl/sac_cont_v1'; PPOC_RUN = f'{DRIVE}/rl_runs_ahr_rl/ppo_cont_v1'
    !python -m ahr_rl.continuous --algo sac --tapes "/content/tapes/*.npz" --out "{SAC_RUN}" --total-steps 500000 --n-envs {max(4, NCPU)} --update-every 2 --eval-races 200
    !python -m ahr_rl.continuous --algo ppo --tapes "/content/tapes/*.npz" --out "{PPOC_RUN}" --total-steps 2000000 --n-envs {max(4, NCPU)} --eval-races 200
    !python -m ahr_rl.compare --name full --discrete "{RUN}" --ppo-cont "{PPOC_RUN}" --sac "{SAC_RUN}" --out "{DRIVE}/rl_runs_ahr_rl/compare"

## 6. Paper-trade live
On a machine with Betfair API credentials (not Colab):
```
export BF_USERNAME=... BF_PASSWORD=... BF_APP_KEY=... BF_CERTS=/path/to/certs
python -m ahr_rl.live.bot --model runs/ppo_v1/best.pt          # paper (default)
```

# V2 agent (built from everything the tests showed)
See `docs/V2.md`. Run cell 1 (tapes) first in a fresh session. Order: V2.1 -> V2.2 -> V2.3 -> V2.4.
The edge-test samples from cell 2b (and the long-horizon run, if you did it) are reused by the forecaster.

## V2.1 Attach catalogue form features to the tapes (once; cached on Drive)

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
EDGE_LONG = f'{DRIVE}/rl_runs_ahr_rl/edge_long'
FC = f'{DRIVE}/rl_runs_ahr_rl/forecaster_v2.pkl'
RUN_V2 = f'{DRIVE}/rl_runs_ahr_rl/ppo_v2'
NCPU = os.cpu_count()
!python -m ahr_rl.catalogue "{TAPES}" "{DRIVE}/catalogues"
!rsync -a "{TAPES}/" /content/tapes/

## V2.2 Fit the cross-fitted price-move forecaster (train days only)

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
EDGE_LONG = f'{DRIVE}/rl_runs_ahr_rl/edge_long'
FC = f'{DRIVE}/rl_runs_ahr_rl/forecaster_v2.pkl'
RUN_V2 = f'{DRIVE}/rl_runs_ahr_rl/ppo_v2'
NCPU = os.cpu_count()
samples = [f'{EDGE}/samples_base.parquet'] + ([f'{EDGE_LONG}/samples_base.parquet'] if os.path.exists(f'{EDGE_LONG}/samples_base.parquet') else [])
SAMPLES = ",".join(samples)
print(SAMPLES)
!python -m ahr_rl.forecaster --samples "{SAMPLES}" --tapes "/content/tapes/*.npz" --out "{FC}"

## V2.3 Train the v2 agent
10s decisions, trades held for minutes or to the start, v2 features + forecaster, exploration floor.
~3M steps; watch `VAL green` in the log. `do nothing` = 0 is the bar.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
EDGE_LONG = f'{DRIVE}/rl_runs_ahr_rl/edge_long'
FC = f'{DRIVE}/rl_runs_ahr_rl/forecaster_v2.pkl'
RUN_V2 = f'{DRIVE}/rl_runs_ahr_rl/ppo_v2'
NCPU = os.cpu_count()
!python -m ahr_rl.train --env v2 --forecaster "{FC}" --tapes "/content/tapes/*.npz" --out "{RUN_V2}" --total-steps 3000000 --n-envs {max(4, NCPU)} --n-steps 256 --explore-floor 0.05 --explore-anneal-frac 0.5 --type-ent-coef 0.01 --ent-coef 0.003 --eval-every 25 --eval-races 60

## V2.4 Held-out test report, segments and deployment gate
Writes `gate.json`; the live bot refuses real money unless `passed` is true.

In [ ]:
import os
DRIVE = '/content/drive/MyDrive/betfair stream data'
TAPES = f'{DRIVE}/rl_tapes_v1'
EDGE = f'{DRIVE}/rl_runs_ahr_rl/edge'
EDGE_LONG = f'{DRIVE}/rl_runs_ahr_rl/edge_long'
FC = f'{DRIVE}/rl_runs_ahr_rl/forecaster_v2.pkl'
RUN_V2 = f'{DRIVE}/rl_runs_ahr_rl/ppo_v2'
NCPU = os.cpu_count()
!python -m ahr_rl.report --run "{RUN_V2}" --split test